# 🌌 Kepler Exoplanet Transit Detection
## Data Cleaning → EDA → Machine Learning Pipeline

**Dataset:** Kepler Objects of Interest (KOI) — NASA Exoplanet Archive

> Run cells one by one with **Shift + Enter**. Every cell has a comment explaining what it does.


---
## 0. Import Libraries

In [ ]:
# Uncomment the line below if running on Google Colab
# !pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, RocCurveDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print("All libraries imported successfully!")

---
## 1. Load the Dataset

📝 Make sure `kepler_transit_detection_messy.csv` is in the same folder as this notebook.

In [ ]:
# Change the filename below if yours is named differently
FILE = 'kepler_transit_detection_messy.csv'

df_raw = pd.read_csv(FILE)

print(f"Rows: {df_raw.shape[0]}  |  Columns: {df_raw.shape[1]}")
print()
df_raw.head(3)

In [ ]:
# Quick look at data types and missing value counts
df_raw.info()

---
## 2. Explore — Find the Data Problems

Before cleaning anything, let's see exactly what is broken.

In [ ]:
# ── 2.1  How many values are missing per column? ──────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"Columns with missing data: {len(missing_df)} out of {len(df_raw.columns)}")
print()
print(missing_df.to_string())

In [ ]:
# ── 2.2  Visualise missing values as a heatmap ────────────────────────────────
# Yellow = missing cell, dark = present
top_cols = missing_df.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(df_raw[top_cols].isnull(),
            yticklabels=False, cbar=False, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Value Map (top 20 columns)  —  Yellow = missing', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('missing_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.3  Check the target column (koi_disposition) ───────────────────────────
# It should only have 3 values: CONFIRMED, FALSE POSITIVE, CANDIDATE
# But the messy data has many spelling variants!
print("Unique values in koi_disposition:")
print(df_raw['koi_disposition'].value_counts().to_string())
print()
print(f"Total unique variants: {df_raw['koi_disposition'].nunique()}")
print()
print("Problem: 'CONFIRMED', 'confirmed', 'Confirm', 'Yes' all mean the same thing.")
print("We need to standardise all of these to one label.")

In [ ]:
# ── 2.4  Find physically impossible values ────────────────────────────────────
print("=== Physically Impossible Values ===")

checks = {
    'koi_period  (negative period)' : lambda s: (s < 0).sum(),
    'koi_score   (score > 1.0)'     : lambda s: (s > 1).sum(),
    'koi_steff   (temp <= 0 K)'     : lambda s: (s <= 0).sum(),
    'ra          (RA > 360 deg)'    : lambda s: (s > 360).sum(),
    'dec         (Dec > 90 deg)'    : lambda s: (s > 90).sum(),
}
for col_label, check_fn in checks.items():
    col = col_label.split()[0]
    series = pd.to_numeric(df_raw[col], errors='coerce')
    print(f"  {col_label:35s}: {check_fn(series)} rows")

In [ ]:
# ── 2.5  Find string garbage inside numeric columns ───────────────────────────
# Some cells have "N/A", "--", "?" instead of a number
print("=== String Garbage in Numeric Columns ===")
check_cols = ['koi_period', 'koi_duration', 'koi_depth',
              'koi_snr', 'koi_steff', 'koi_prad']

for col in check_cols:
    s = df_raw[col].astype(str)
    bad = s[pd.to_numeric(s, errors='coerce').isna() & (s != 'nan') & s.notna()]
    if len(bad) > 0:
        print(f"  {col:20s}: {len(bad)} bad values — e.g. {bad.unique()[:4].tolist()}")

print()
print("These will be converted to NaN during cleaning.")

In [ ]:
# ── 2.6  Check for duplicate Kepler IDs ──────────────────────────────────────
dup_count = df_raw.duplicated(subset=['kepid']).sum()
print(f"Duplicate kepid entries : {dup_count}")
print(f"Total rows              : {len(df_raw)}")
print(f"Unique kepids           : {df_raw['kepid'].nunique()}")
print()
print("Duplicates must be removed to prevent data leakage.")

---
## 3. Data Cleaning — 8 Steps

We fix every problem found in Section 2. Always work on a **copy** — never touch df_raw.


In [ ]:
# ── Always start with a copy ──────────────────────────────────────────────────
df = df_raw.copy()
print(f"Working copy created: {df.shape}")

In [ ]:
# ── Step 1: Standardise target labels ─────────────────────────────────────────
label_map = {
    'CONFIRMED': 'CONFIRMED', 'confirmed': 'CONFIRMED',
    'Confirmed': 'CONFIRMED', 'CONFIRM': 'CONFIRMED',
    'confirm':   'CONFIRMED', 'Yes':      'CONFIRMED',

    'FALSE POSITIVE': 'FALSE POSITIVE', 'false positive': 'FALSE POSITIVE',
    'False Positive': 'FALSE POSITIVE', 'FALSE_POSITIVE': 'FALSE POSITIVE',
    'false_positive': 'FALSE POSITIVE', 'FP': 'FALSE POSITIVE', 'fp': 'FALSE POSITIVE',

    'CANDIDATE': 'CANDIDATE', 'candidate': 'CANDIDATE',
    'Candidate': 'CANDIDATE', 'CAND':      'CANDIDATE',
    'cand':      'CANDIDATE', 'Maybe':     'CANDIDATE',
}

df['koi_disposition'] = df['koi_disposition'].map(label_map)
before = len(df)
df = df.dropna(subset=['koi_disposition'])

print(f"Rows dropped (unmappable label): {before - len(df)}")
print()
print("Label distribution after step 1:")
print(df['koi_disposition'].value_counts().to_string())

In [ ]:
# ── Step 2: Convert string garbage to NaN ─────────────────────────────────────
# pd.to_numeric(errors='coerce') turns "N/A", "?", "--" etc. into NaN
numeric_cols = [
    'koi_period', 'koi_period_err1', 'koi_period_err2',
    'koi_time0bk', 'koi_impact', 'koi_duration',
    'koi_duration_err1', 'koi_duration_err2',
    'koi_depth', 'koi_depth_err1', 'koi_depth_err2',
    'koi_prad', 'koi_prad_err1', 'koi_prad_err2',
    'koi_sma', 'koi_teq', 'koi_insol',
    'koi_steff', 'koi_steff_err1', 'koi_steff_err2',
    'koi_slogg', 'koi_slogg_err1', 'koi_slogg_err2',
    'koi_srad', 'koi_srad_err1', 'koi_srad_err2',
    'koi_smass', 'koi_smet', 'koi_sage',
    'koi_snr', 'koi_score', 'ra', 'dec', 'koi_kepmag'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"String garbage converted to NaN in {len(numeric_cols)} columns.")

In [ ]:
# ── Step 3: Remove physically impossible values ────────────────────────────────
before = len(df)

df = df[~((df['koi_period'] < 0)    & df['koi_period'].notna())]
df = df[~((df['koi_score']  > 1.0)  & df['koi_score'].notna())]
df = df[~((df['koi_steff']  <= 0)   & df['koi_steff'].notna())]
df = df[~((df['ra']          > 360)  & df['ra'].notna())]
df = df[~((df['dec']         > 90)   & df['dec'].notna())]

print(f"Impossible values removed: {before - len(df)} rows")
print(f"Remaining rows: {len(df)}")

In [ ]:
# ── Step 4: Fix unit mismatch in koi_duration ─────────────────────────────────
# About 7% of values were recorded in MINUTES instead of HOURS.
# Any transit > 15 hours is impossible, so those are in minutes — divide by 60.
mask = (df['koi_duration'] > 15.0) & df['koi_duration'].notna()
df.loc[mask, 'koi_duration'] = df.loc[mask, 'koi_duration'] / 60.0

print(f"Duration unit correction applied to {mask.sum()} rows.")
print(f"Duration range now: {df['koi_duration'].min():.2f} — {df['koi_duration'].max():.2f} hours")

In [ ]:
# ── Step 5: Remove duplicate Kepler IDs ───────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=['kepid'], keep='first')
print(f"Duplicates removed: {before - len(df)}")
print(f"Remaining rows: {len(df)}")

In [ ]:
# ── Step 6: Fix false positive flags (must be 0 or 1 only) ───────────────────
flag_cols = ['koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']

for col in flag_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].where(df[col].isin([0, 1]), other=np.nan)
        df[col] = df[col].fillna(0).astype(int)

print("False positive flags fixed to 0/1.")

In [ ]:
# ── Step 7: Handle missing values ─────────────────────────────────────────────
# Drop columns with > 40% missing (too unreliable to fill)
# Fill the rest with the column median

threshold = 0.40
cols_to_drop = [c for c in numeric_cols
                if c in df.columns and df[c].isnull().mean() > threshold]
df = df.drop(columns=cols_to_drop)
print(f"Columns dropped (> 40% missing): {cols_to_drop}")

# Fill remaining missing values with column median
remaining = [c for c in numeric_cols if c in df.columns]
for col in remaining:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print(f"Remaining missing values: {df[remaining].isnull().sum().sum()}")

In [ ]:
# ── Step 8: Summary + save ────────────────────────────────────────────────────
print("=" * 45)
print("  CLEANING SUMMARY")
print("=" * 45)
print(f"  Original rows  : {len(df_raw)}")
print(f"  After cleaning : {len(df)}")
print(f"  Rows removed   : {len(df_raw) - len(df)}")
print()
print("  Final class distribution:")
print(df['koi_disposition'].value_counts().to_string())
print()
print(f"  Final shape: {df.shape}")
print("=" * 45)

df.to_csv('kepler_cleaned.csv', index=False)
print("Clean data saved as kepler_cleaned.csv")

---
## 4. Exploratory Data Analysis (EDA)

Six charts to understand our data. These go directly into your research paper.


In [ ]:
# ── 4.1  Class distribution — the imbalance problem ──────────────────────────
counts = df['koi_disposition'].value_counts()
colors = ['#1D9E75', '#E24B4A', '#EF9F27']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Objects')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 8, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.suptitle('Class Imbalance in the Kepler KOI Dataset', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()
print("Saved as class_distribution.png")
print()
print("KEY INSIGHT: Only ~35% are confirmed planets.")
print("A model predicting 'not a planet' every time would score ~65% accuracy!")
print("This is WHY we need SMOTE and imbalance-aware metrics.")

In [ ]:
# ── 4.2  KDE distributions — do features separate the classes? ───────────────
features_to_plot = {
    'koi_period'  : 'Orbital Period (days)',
    'koi_duration': 'Transit Duration (hours)',
    'koi_depth'   : 'Transit Depth (ppm)',
    'koi_prad'    : 'Planet Radius (Earth radii)',
    'koi_snr'     : 'Signal-to-Noise Ratio',
    'koi_steff'   : 'Stellar Temperature (K)',
}
palette = {'CONFIRMED': '#1D9E75', 'FALSE POSITIVE': '#E24B4A', 'CANDIDATE': '#EF9F27'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (col, label) in zip(axes, features_to_plot.items()):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    for disposition, color in palette.items():
        subset = df[df['koi_disposition'] == disposition][col].dropna()
        # Clip top 1% outliers just for the visualisation
        subset = subset[subset <= subset.quantile(0.99)]
        subset.plot.kde(ax=ax, label=disposition, color=color, linewidth=2)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.set_xlim(left=0)

plt.suptitle('Feature Distributions by Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()
print("Saved as feature_distributions.png")

In [ ]:
# ── 4.3  Correlation heatmap ──────────────────────────────────────────────────
# Select the 14 features we will use for ML
ml_features = [
    'koi_period', 'koi_duration', 'koi_depth', 'koi_prad',
    'koi_impact', 'koi_snr', 'koi_score',
    'koi_steff', 'koi_slogg', 'koi_srad',
    'koi_teq', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co'
]
# Keep only features that survived cleaning
ml_features = [f for f in ml_features if f in df.columns]
print(f"ML features available: {len(ml_features)}")

corr = df[ml_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide the redundant upper triangle

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved as correlation_heatmap.png")

In [ ]:
# ── 4.4  Boxplots — spread and outliers by class ─────────────────────────────
plot_cols = ['koi_period', 'koi_snr', 'koi_depth', 'koi_prad', 'koi_steff', 'koi_srad']
plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors_bp = ['#1D9E75', '#E24B4A', '#EF9F27']

for ax, col in zip(axes, plot_cols):
    data_by_class = [
        df[df['koi_disposition'] == d][col].dropna().values
        for d in ['CONFIRMED', 'FALSE POSITIVE', 'CANDIDATE']
    ]
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    labels=['CONF', 'FP', 'CAND'], showfliers=False)
    for patch, color in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Boxplots by Class (outliers hidden)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', bbox_inches='tight')
plt.show()
print("Saved as boxplots.png")

In [ ]:
# ── 4.5  Scatter: Orbital Period vs Planet Radius ─────────────────────────────
palette = {'CONFIRMED': '#1D9E75', 'FALSE POSITIVE': '#E24B4A', 'CANDIDATE': '#EF9F27'}

fig, ax = plt.subplots(figsize=(9, 6))
for disposition, color in palette.items():
    subset = df[df['koi_disposition'] == disposition]
    ax.scatter(subset['koi_period'], subset['koi_prad'],
               alpha=0.4, s=15, color=color, label=disposition)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Orbital Period (days) — log scale')
ax.set_ylabel('Planet Radius (Earth radii) — log scale')
ax.set_title('Orbital Period vs Planet Radius by Class', fontweight='bold')
ax.axhline(y=1.0,  color='gray', linestyle='--', linewidth=0.8)
ax.axhline(y=11.2, color='navy', linestyle='--', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.savefig('period_vs_radius.png', bbox_inches='tight')
plt.show()
print("Saved as period_vs_radius.png")

---
## 5. Prepare Data for Machine Learning


In [ ]:
# ── 5.1  Create binary target: CONFIRMED = 1, everything else = 0 ─────────────
df_ml = df[ml_features + ['koi_disposition']].copy()
df_ml['target'] = (df_ml['koi_disposition'] == 'CONFIRMED').astype(int)
df_ml = df_ml.drop(columns=['koi_disposition'])

print("Target distribution:")
print(df_ml['target'].value_counts()
      .rename({1: 'CONFIRMED (1)', 0: 'NOT CONFIRMED (0)'}).to_string())
print()
print(f"Positive class ratio: {df_ml['target'].mean():.2%}")

In [ ]:
# ── 5.2  Stratified train/test split (80/20) ──────────────────────────────────
# stratify=y ensures both sets have the same class ratio
X = df_ml.drop(columns=['target'])
y = df_ml['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {len(X_train)} samples  ({y_train.mean():.2%} positive)")
print(f"Test set     : {len(X_test)} samples  ({y_test.mean():.2%} positive)")

In [ ]:
# ── 5.3  Feature scaling (fit on train only — never on test!) ─────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from training
X_test_scaled  = scaler.transform(X_test)          # apply same scale to test

print(f"Training mean  ≈ {X_train_scaled.mean():.4f}  (should be ~0)")
print(f"Training std   ≈ {X_train_scaled.std():.4f}  (should be ~1)")

In [ ]:
# ── 5.4  Apply SMOTE to training set only ─────────────────────────────────────
# SMOTE creates synthetic minority-class samples to balance the classes
# IMPORTANT: ONLY apply to training data, NEVER to test data
print("Before SMOTE:")
print(pd.Series(y_train).value_counts()
      .rename({1: 'CONFIRMED', 0: 'NOT CONFIRMED'}).to_string())

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print()
print("After SMOTE:")
print(pd.Series(y_train_sm).value_counts()
      .rename({1: 'CONFIRMED', 0: 'NOT CONFIRMED'}).to_string())
print()
print(f"Training size: {len(X_train_scaled)} → {len(X_train_sm)} (after SMOTE)")

---
## 6. Train and Evaluate 5 Classifiers

Each classifier is trained twice — once without SMOTE, once with SMOTE.
We use F1-Score, Precision, Recall, and AUC-ROC (NOT accuracy — accuracy is misleading on imbalanced data).


In [ ]:
# ── 6.1  Define the 5 classifiers ─────────────────────────────────────────────
classifiers = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, random_state=42, class_weight='balanced'),

    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42,
        class_weight='balanced', n_jobs=-1),

    'SVM': SVC(
        kernel='rbf', C=1.0, probability=True,
        random_state=42, class_weight='balanced'),

    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42, eval_metric='logloss', verbosity=0),
}

print("Classifiers ready:")
for name in classifiers:
    print(f"  • {name}")

In [ ]:
# ── 6.2  Train and evaluate each classifier (with and without SMOTE) ──────────
results = []

for name, clf in classifiers.items():
    print(f"Training {name}...", end=' ', flush=True)
    row = {'Classifier': name}

    # --- Train WITHOUT SMOTE ---
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    row['F1 (no SMOTE)']        = round(f1_score(y_test, y_pred, zero_division=0), 3)
    row['Precision (no SMOTE)'] = round(precision_score(y_test, y_pred, zero_division=0), 3)
    row['Recall (no SMOTE)']    = round(recall_score(y_test, y_pred, zero_division=0), 3)
    row['AUC (no SMOTE)']       = round(roc_auc_score(y_test, y_prob), 3)

    # --- Train WITH SMOTE ---
    clf.fit(X_train_sm, y_train_sm)
    y_pred_sm = clf.predict(X_test_scaled)
    y_prob_sm = clf.predict_proba(X_test_scaled)[:, 1]

    row['F1 (SMOTE)']        = round(f1_score(y_test, y_pred_sm, zero_division=0), 3)
    row['Precision (SMOTE)'] = round(precision_score(y_test, y_pred_sm, zero_division=0), 3)
    row['Recall (SMOTE)']    = round(recall_score(y_test, y_pred_sm, zero_division=0), 3)
    row['AUC (SMOTE)']       = round(roc_auc_score(y_test, y_prob_sm), 3)

    results.append(row)
    print("done")

results_df = pd.DataFrame(results)
print()
print("=" * 80)
print("RESULTS TABLE  (copy this into your research paper)")
print("=" * 80)
print(results_df.to_string(index=False))

In [ ]:
# ── 6.3  F1-Score comparison bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(results_df))
w = 0.35

bars1 = ax.bar(x - w/2, results_df['F1 (no SMOTE)'], w,
               label='Without SMOTE', color='#B5D4F4', edgecolor='white')
bars2 = ax.bar(x + w/2, results_df['F1 (SMOTE)'], w,
               label='With SMOTE', color='#1D9E75', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_xlabel('Classifier')
ax.set_ylabel('F1-Score (Planet Class)')
ax.set_title('F1-Score Comparison: With vs Without SMOTE', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Classifier'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.08)
plt.tight_layout()
plt.savefig('f1_comparison.png', bbox_inches='tight')
plt.show()
print("Saved as f1_comparison.png")

In [ ]:
# ── 6.4  ROC Curves (with SMOTE) ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#534AB7', '#1D9E75', '#D85A30', '#D4537E', '#BA7517']

for (name, clf), color in zip(classifiers.items(), colors_roc):
    clf.fit(X_train_sm, y_train_sm)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=f"{name} (AUC={auc:.3f})",
        ax=ax, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random Classifier')
ax.set_title('ROC Curves (with SMOTE)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()
print("Saved as roc_curves.png")

In [ ]:
# ── 6.5  Confusion matrices (with SMOTE) ─────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, clf) in zip(axes, classifiers.items()):
    clf.fit(X_train_sm, y_train_sm)
    y_pred = clf.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Planet', 'Planet'],
                yticklabels=['Not Planet', 'Planet'],
                cbar=False)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — With SMOTE', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()
print("Saved as confusion_matrices.png")

---
## 7. Feature Importance — SHAP Analysis

SHAP tells us **which features** matter most and **why** the model made each prediction.


In [ ]:
# ── 7.1  Find the best classifier by F1 (SMOTE) ──────────────────────────────
best_row  = results_df.loc[results_df['F1 (SMOTE)'].idxmax()]
best_name = best_row['Classifier']
print(f"Best classifier: {best_name}")
print(f"F1 Score (with SMOTE): {best_row['F1 (SMOTE)']}")

In [ ]:
# ── 7.2  Retrain best model and compute SHAP values ───────────────────────────
best_clf = classifiers[best_name]
best_clf.fit(X_train_sm, y_train_sm)

# TreeExplainer works for Random Forest, XGBoost, Decision Tree
# KernelExplainer works for everything else (slower)
try:
    explainer  = shap.TreeExplainer(best_clf)
    shap_vals  = explainer.shap_values(X_test_scaled)
    # For binary classification, shap_values returns [class0, class1]
    # We want class 1 (planet = positive class)
    if isinstance(shap_vals, list):
        sv = shap_vals[1]
    else:
        sv = shap_vals
    print("SHAP TreeExplainer used.")
except Exception:
    explainer = shap.KernelExplainer(best_clf.predict_proba, X_train_sm[:100])
    shap_vals = explainer.shap_values(X_test_scaled[:100])
    sv = shap_vals[1]
    print("SHAP KernelExplainer used.")

print(f"SHAP values shape: {np.array(sv).shape}")

In [ ]:
# ── 7.3  Global feature importance bar chart ──────────────────────────────────
feature_names = list(X.columns)

# Average absolute SHAP value per feature = overall importance
mean_shap = np.abs(sv).mean(axis=0)

shap_df = pd.DataFrame({
    'Feature':    feature_names,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(shap_df['Feature'], shap_df['Mean |SHAP|'],
               color='#378ADD', edgecolor='white')

# Highlight top 3 in dark blue
top3 = shap_df.nlargest(3, 'Mean |SHAP|')['Feature'].values
for i, feat in enumerate(shap_df['Feature']):
    if feat in top3:
        bars[i].set_color('#1F4E79')

ax.set_xlabel('Mean |SHAP Value|  (average impact on model output)')
ax.set_title('Feature Importance - ' + best_name + chr(10) + 'Dark blue = top 3 most important features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', bbox_inches='tight')
plt.show()
print("Saved as shap_importance.png")
print()
print("Top 5 most important features:")
print(shap_df.tail(5)[['Feature', 'Mean |SHAP|']].iloc[::-1].to_string(index=False))

In [ ]:
# ── 7.4  SHAP Beeswarm plot ────────────────────────────────────────────────────
# Shows both importance AND direction of each feature's effect.
# Red = high feature value pushes prediction toward CONFIRMED (planet)
# Blue = high feature value pushes prediction away from CONFIRMED
plt.figure(figsize=(9, 6))
shap.summary_plot(
    sv,
    X_test_scaled,
    feature_names=feature_names,
    plot_type='dot',
    show=False,
    max_display=14
)
plt.title('SHAP Beeswarm Plot - ' + best_name, fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', bbox_inches='tight')
plt.show()
print("Saved as shap_beeswarm.png")

---
## 8. Final Results Summary

Run this cell last. Copy the output into your research paper's Results section.


In [ ]:
# ── Final summary — copy this into your research paper ────────────────────────
print("=" * 65)
print("  RESEARCH PAPER — RESULTS SUMMARY")
print("=" * 65)
print()
print(f"  Dataset (after cleaning)")
print(f"    Total samples     : {len(df_ml)}")
print(f"    Confirmed planets : {(df_ml['target']==1).sum()}  ({df_ml['target'].mean():.1%})")
print(f"    Non-planet        : {(df_ml['target']==0).sum()}  ({1 - df_ml['target'].mean():.1%})")
print(f"    Features used     : {len(ml_features)}")
print(f"    Train / Test      : 80% / 20% (stratified)")
print()
print("-" * 65)
print("  CLASSIFIER COMPARISON (F1-Score, Planet Class)")
print("-" * 65)

for _, row in results_df.sort_values('F1 (SMOTE)', ascending=False).iterrows():
    delta = row['F1 (SMOTE)'] - row['F1 (no SMOTE)']
    sign  = '+' if delta >= 0 else ''
    print(f"  {row['Classifier']:22s}"
          f"  Before SMOTE: {row['F1 (no SMOTE)']:.3f}"
          f"  After SMOTE: {row['F1 (SMOTE)']:.3f}"
          f"  Change: {sign}{delta:.3f}")

print()
print("-" * 65)
print("  BEST CLASSIFIER")
print("-" * 65)
best = results_df.loc[results_df['F1 (SMOTE)'].idxmax()]
print(f"  {best['Classifier']}")
print(f"    F1-Score  : {best['F1 (SMOTE)']:.4f}")
print(f"    Precision : {best['Precision (SMOTE)']:.4f}")
print(f"    Recall    : {best['Recall (SMOTE)']:.4f}")
print(f"    AUC-ROC   : {best['AUC (SMOTE)']:.4f}")
print()
print("-" * 65)
print("  TOP 3 MOST IMPORTANT FEATURES (SHAP)")
print("-" * 65)
top_feat_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP':    np.abs(sv).mean(axis=0)
}).sort_values('SHAP', ascending=False).head(3)

for i, (_, row) in enumerate(top_feat_df.iterrows(), 1):
    print(f"  {i}. {row['Feature']:22s}  Mean |SHAP| = {row['SHAP']:.4f}")

print()
print("=" * 65)
print("  FIGURES SAVED:")
figs = [
    'missing_heatmap.png', 'class_distribution.png',
    'feature_distributions.png', 'correlation_heatmap.png',
    'boxplots.png', 'period_vs_radius.png',
    'f1_comparison.png', 'roc_curves.png',
    'confusion_matrices.png', 'shap_importance.png',
    'shap_beeswarm.png'
]
for f in figs:
    print(f"  • {f}")
print("=" * 65)

---
## 9. Next Steps

1. Copy the results table from Section 6.2 into **Table I** of your research paper.
2. Insert the figures into **Section 5 (Results)** of your paper.
3. Write 2–3 sentences under each figure explaining what it shows.
4. Replace the placeholder numbers in **Section 6 (Discussion)** with your real results.
5. Push this notebook to GitHub and add the link to your paper's Conclusion.
